# Day 17 – Working with Local Models – Extensive Solutions

Complete solutions for running Llama 3, Mistral locally via Ollama, llama.cpp, and Hugging Face transformers with 4-bit quantisation.

## Setup: Install required tools

In [ ]:
# For Ollama (install system-wide first: https://ollama.com)
# !curl -fsSL https://ollama.com/install.sh | sh
# !ollama pull llama3

# For transformers with bitsandbytes
# !pip install transformers accelerate bitsandbytes torch

print("Make sure Ollama is installed and running.")

## Exercise 1: Run Llama 3 on CPU using Ollama

We'll use Ollama's HTTP API to generate text.

In [ ]:
import requests
import json

def ollama_generate(prompt, model="llama3", temperature=0.7, max_tokens=100):
    """Generate text using Ollama's API."""
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False
    }
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        return response.json()["response"]
    except requests.exceptions.ConnectionError:
        return "Error: Ollama not running. Start with 'ollama serve'"

# Test
output = ollama_generate("What is the capital of France?", temperature=0)
print(output)

## Exercise 2: Compare response quality between Llama 3 and GPT-3.5

We'll ask the same 5 questions to both models and compare manually (or use BERTScore).

In [ ]:
import openai
from dotenv import load_dotenv
import os
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

def gpt35_generate(prompt, temp=0):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp
    )
    return response.choices[0].message.content

questions = [
    "Explain quantum computing in one sentence.",
    "What is the tallest mountain in the world?",
    "Write a haiku about autumn.",
    "What is the capital of Brazil?",
    "Explain the theory of relativity simply."
]

comparison = []
for q in questions:
    print(f"\n--- Question: {q} ---")
    gpt_answer = gpt35_generate(q, temp=0)
    llama_answer = ollama_generate(q, temperature=0)
    print(f"GPT-3.5: {gpt_answer}\n")
    print(f"Llama 3: {llama_answer}\n")
    comparison.append((q, gpt_answer, llama_answer))

# Optional: compute BERTScore between answers (requires reference)
# from evaluate import load
# bertscore = load("bertscore")
# ...

## Exercise 3: Measure inference speed (tokens/second)

We'll generate 100 tokens and measure time.

In [ ]:
import time

def measure_speed(model_func, prompt, max_tokens=100):
    start = time.time()
    output = model_func(prompt, max_tokens=max_tokens)
    elapsed = time.time() - start
    # Approximate token count (words + punctuation)
    tokens = len(output.split())
    speed = tokens / elapsed
    return tokens, elapsed, speed

prompt = "Write a short story about a robot learning to paint."
tokens, elapsed, speed = measure_speed(ollama_generate, prompt, max_tokens=100)
print(f"Ollama (CPU): {tokens} tokens in {elapsed:.2f}s = {speed:.1f} tokens/sec")

# For GPT-3.5 (API, not local)
def gpt_speed(prompt, max_tokens=100):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

tokens_g, elapsed_g, speed_g = measure_speed(gpt_speed, prompt, max_tokens=100)
print(f"GPT-3.5 (API): {tokens_g} tokens in {elapsed_g:.2f}s = {speed_g:.1f} tokens/sec")

## Exercise 4: Fine‑tune locally using Unsloth (optional, GPU required)

Unsloth makes fine‑tuning much faster and uses less memory.

In [ ]:
# !pip install unsloth
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained("unsloth/llama-3-8b-bnb-4bit")
# model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16)
# ... then train on a small dataset
print("Unsloth fine‑tuning requires a GPU and is not run here for brevity.")

## Bonus: Run a GGUF model with llama.cpp

Download a quantised model and run with `llama-cpp-python`.

In [ ]:
# !pip install llama-cpp-python
# from llama_cpp import Llama
# llm = Llama(model_path="path/to/llama3-8b-q4_k_m.gguf", n_ctx=512, n_threads=4)
# output = llm("What is AI?", max_tokens=50)
# print(output["choices"][0]["text"])
print("llama.cpp example – requires downloading a GGUF file first.")